# Telco churn: trees vs tabular Transformers

This notebook trains a **complete** set of models on IBM Telco Customer Churn and compares them on the **same** stratified split.

| Family | Model | Why it belongs here |
|---|---|---|
| Linear baseline | Logistic Regression | Strong, interpretable floor on this table |
| Tree ensembles | Random Forest, XGBoost | Usual winners on medium tabular data |
| Tabular Transformers | **FT-Transformer**, **TabTransformer**, **SAINT** | Attention over *features* (and rows), not over words |

A BERT-style NLP model is the wrong tool: there is no token sequence, no pretrained vocabulary, and only 7,043 rows. The Transformer architectures below treat **each column as a token**.

```
CSV → pandas → shared 70/15/15 split
                ├─ scikit-learn / XGBoost  → P(churn)
                └─ PyTorch Tabular         → FT-Transformer / TabTransformer → P(churn)
                   custom SAINT            → P(churn)
```

Environment: conda `llm10`. Primary ranking metric: **ROC-AUC**. Operating point: validation F1 for the churn class, then frozen on test.


In [ ]:
import sys
import warnings
from pathlib import Path

_pkg = Path.cwd()
if not (_pkg / "tabular_transformer_churn.py").is_file():
    _pkg = _pkg / "tabular_transformers"
sys.path.insert(0, str(_pkg))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
)

from tabular_transformer_churn import (
    CATEGORICAL_COLS,
    DEFAULT_CSV,
    FEATURE_COLS,
    NUMERICAL_COLS,
    OUTPUT_DIR,
    TARGET_COL,
    _binary_labels,
    load_and_prepare,
    run,
    split_frames,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#111827",
    "axes.labelcolor": "#111827",
    "xtick.color": "#111827",
    "ytick.color": "#111827",
    "text.color": "#111827",
    "axes.titlecolor": "#111827",
})
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("csv", DEFAULT_CSV)
print("features", len(FEATURE_COLS), "cat", len(CATEGORICAL_COLS), "num", len(NUMERICAL_COLS))


## 1. Load, clean, and inspect the table

`customerID` is an identifier and is dropped from features. Blank `TotalCharges` are new customers with `tenure == 0`, so they are filled with 0 (not the column mean). `SeniorCitizen` is treated as categorical, which matters for TabTransformer.


In [ ]:
data = load_and_prepare(DEFAULT_CSV)
train, val, test = split_frames(data, seed=42)
y_train = _binary_labels(train[TARGET_COL])
y_val = _binary_labels(val[TARGET_COL])
y_test = _binary_labels(test[TARGET_COL])

print("shape", data.shape)
print("churn rate {:.1%}".format(data[TARGET_COL].eq("Yes").mean()))
print("split train/val/test", len(train), len(val), len(test))
print("churn rate by split",
      float(train[TARGET_COL].eq("Yes").mean()),
      float(val[TARGET_COL].eq("Yes").mean()),
      float(test[TARGET_COL].eq("Yes").mean()))
print("missing after clean", int(data[FEATURE_COLS].isna().sum().sum()))
display(data[FEATURE_COLS + [TARGET_COL]].head())
display(data[TARGET_COL].value_counts().to_frame("count"))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
sns.countplot(data=data, x="Churn", ax=axes[0], color="#2563EB")
axes[0].set_title("Churn class balance")
sns.barplot(
    data=data.assign(churn=data[TARGET_COL].eq("Yes").astype(int)),
    x="Contract", y="churn", ax=axes[1], color="#2563EB", errorbar=None,
)
axes[1].set_ylabel("Churn rate")
axes[1].set_title("Churn rate by contract")
sns.kdeplot(data=data, x="tenure", hue="Churn", ax=axes[2], common_norm=False)
axes[2].set_title("Tenure by churn")
fig.suptitle("IBM Telco Customer Churn · n=7,043", y=1.04, fontsize=12)
fig.tight_layout()
plt.show()


## 2. Tabular-specific Transformer architectures

These are not BERT. BERT attends over **word tokens**. These models attend over **table columns**.

**FT-Transformer** (Gorishniy et al., 2021) — *Feature Tokenizer + Transformer*. Every column, categorical and numeric, becomes a d-dimensional token. A CLS token is prepended. Stacked Transformer blocks mix information across features. The CLS vector goes to a linear head that outputs churn logits. This is the default tabular Transformer in this notebook (PyTorch Tabular `FTTransformerConfig`: dim 32, 3 blocks, 4 heads, GEGLU).

**TabTransformer** (Huang et al., 2020) — attention is applied **only to categorical embeddings**, so values become contextual (for example month-to-month given fiber and electronic check). Numeric columns skip the Transformer and are concatenated into an MLP head. Strong when categoricals carry the interactions.

**SAINT** (Somepalli et al., 2021) — feature attention **plus inter-sample (row) attention** inside a mini-batch. PyTorch Tabular does not ship SAINT, so `saint_tabular.py` implements both attention types. At single-customer scoring, row attention is turned **off** so a prediction does not depend on whoever else is in the batch.

Tree models (Random Forest, XGBoost) remain the practical baseline on this size of table. The question this notebook answers is whether a tabular Transformer beats them on Telco churn, not whether a language model can be forced onto a spreadsheet.


## 3. Train every model on the same split

All six models see the same 4,930 / 1,056 / 1,057 rows. Class imbalance is handled with `class_weight` / `scale_pos_weight` / weighted cross-entropy. The decision threshold is chosen on **validation F1 for churners**, then applied once to test. ROC-AUC and average precision do not depend on that threshold.


In [ ]:
# Trains logistic regression, Random Forest, XGBoost, FT-Transformer,
# TabTransformer, and SAINT. A few minutes on Apple MPS.
metrics = run([
    "--models", "logreg,random_forest,xgboost,ft_transformer,tab_transformer,saint",
    "--epochs", "20",
])
display(metrics)


## 4. Leaderboard


In [ ]:
metrics_path = OUTPUT_DIR / "tabular_transformer_metrics.csv"
pred_path = OUTPUT_DIR / "tabular_transformer_test_predictions.csv"
metrics = pd.read_csv(metrics_path).sort_values("roc_auc", ascending=False).reset_index(drop=True)
preds = pd.read_csv(pred_path)
y_true = preds["y_true"].to_numpy()

show = metrics[[
    "model", "library", "roc_auc", "avg_precision", "f1_yes",
    "recall_yes", "precision_yes", "accuracy", "threshold",
    "tn", "fp", "fn", "tp",
]].copy()
for col in ["roc_auc", "avg_precision", "f1_yes", "recall_yes", "precision_yes", "accuracy", "threshold"]:
    show[col] = show[col].map(lambda x: round(float(x), 3))
display(show)

best = metrics.iloc[0]
print(
    "Best by ROC-AUC: {} ({:.3f}). Best tree: {}. Best Transformer: {}.".format(
        best["model"],
        best["roc_auc"],
        metrics.loc[metrics["model"].isin(["Random Forest", "XGBoost"])].iloc[0]["model"],
        metrics.loc[metrics["model"].isin(["FT-Transformer", "TabTransformer", "SAINT"])].sort_values("roc_auc", ascending=False).iloc[0]["model"],
    )
)


In [ ]:
prob_map = {
    "Logistic Regression": "prob_logreg",
    "Random Forest": "prob_random_forest",
    "XGBoost": "prob_xgboost",
    "FT-Transformer": "prob_ft_transformer",
    "TabTransformer": "prob_tab_transformer",
    "SAINT": "prob_saint",
}

fig, ax = plt.subplots(figsize=(8.2, 5.4))
for _, row in metrics.sort_values("roc_auc", ascending=False).iterrows():
    col = prob_map[row["model"]]
    fpr, tpr, _ = roc_curve(y_true, preds[col])
    ax.plot(fpr, tpr, label=f"{row['model']} (AUC={row['roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="#9CA3AF", label="Chance")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("Test ROC — trees vs tabular Transformers")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 5.4))
for _, row in metrics.sort_values("avg_precision", ascending=False).iterrows():
    col = prob_map[row["model"]]
    prec, rec, _ = precision_recall_curve(y_true, preds[col])
    ap = average_precision_score(y_true, preds[col])
    ax.plot(rec, prec, label=f"{row['model']} (AP={ap:.3f})")
base = y_true.mean()
ax.axhline(base, linestyle="--", color="#9CA3AF", label=f"Prevalence ({base:.2f})")
ax.set_xlabel("Recall (churn = Yes)")
ax.set_ylabel("Precision (churn = Yes)")
ax.set_title("Test precision-recall — churn class")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()


In [ ]:
plot_df = metrics.melt(
    id_vars=["model"],
    value_vars=["roc_auc", "avg_precision", "f1_yes", "recall_yes"],
    var_name="metric",
    value_name="value",
)
plot_df["metric"] = plot_df["metric"].map({
    "roc_auc": "ROC-AUC",
    "avg_precision": "Average precision",
    "f1_yes": "F1 (churn)",
    "recall_yes": "Recall (churn)",
})
fig, ax = plt.subplots(figsize=(9.5, 4.6))
sns.barplot(data=plot_df, x="model", y="value", hue="metric", ax=ax)
ax.set_ylim(0.5, 0.9)
ax.set_xlabel("")
ax.set_ylabel("Score")
ax.set_title("Holdout scores by model")
ax.legend(title="", loc="lower right")
plt.xticks(rotation=20, ha="right")
fig.tight_layout()
plt.show()


## 5. Confusion matrices at the validation-tuned threshold

Each panel uses that model's own F1-optimal threshold from the validation set. Counts are on 1,057 test customers (281 actual churners).


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11.5, 7.2))
order = [
    "Logistic Regression", "Random Forest", "XGBoost",
    "FT-Transformer", "TabTransformer", "SAINT",
]
for ax, name in zip(axes.ravel(), order):
    row = metrics.loc[metrics["model"] == name].iloc[0]
    y_hat = (preds[prob_map[name]] >= row["threshold"]).astype(int)
    cm = confusion_matrix(y_true, y_hat, labels=[0, 1])
    disp = ConfusionMatrixDisplay(cm, display_labels=["Stay", "Churn"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{name}\nAUC {row['roc_auc']:.3f} · F1 {row['f1_yes']:.3f}")
fig.suptitle("Test confusion matrices", y=1.02)
fig.tight_layout()
plt.show()

print("Classification report: best model =", metrics.iloc[0]["model"])
best_name = metrics.iloc[0]["model"]
best_hat = (preds[prob_map[best_name]] >= metrics.iloc[0]["threshold"]).astype(int)
print(classification_report(y_true, best_hat, target_names=["Stay", "Churn"], digits=3))


## 6. What the tree models attend to

Random Forest and XGBoost expose split gain / impurity decrease on the one-hot features. Tabular Transformers mix columns inside attention, so a single number per original column is less direct. Use these plots to sanity-check the table (contract, tenure, fiber, electronic check usually dominate Telco churn).


In [ ]:
rf_imp = pd.read_csv(OUTPUT_DIR / "random_forest_feature_importance.csv").head(12)
xgb_imp = pd.read_csv(OUTPUT_DIR / "xgboost_feature_importance.csv").head(12)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.0))
sns.barplot(data=rf_imp, y="feature", x="importance", ax=axes[0], color="#2563EB")
axes[0].set_title("Random Forest · top 12 features")
sns.barplot(data=xgb_imp, y="feature", x="importance", ax=axes[1], color="#0F766E")
axes[1].set_title("XGBoost · top 12 features")
for ax in axes:
    ax.set_ylabel("")
    ax.set_xlabel("Importance")
fig.suptitle("Tree feature importance (one-hot features)", y=1.02)
fig.tight_layout()
plt.show()


## 7. Score customers: churn probability

The column to use in a retention workflow is **P(churn = Yes)**. Below: the 10 test customers with the highest FT-Transformer score, plus a small unscored-looking sample from the head of the test frame.


In [ ]:
rank = preds.sort_values("prob_ft_transformer", ascending=False).head(10).copy()
keep = ["customerID", "Churn", "prob_xgboost", "prob_random_forest",
        "prob_ft_transformer", "prob_tab_transformer", "prob_saint"]
display(rank[keep].rename(columns={
    "prob_xgboost": "XGBoost",
    "prob_random_forest": "Random Forest",
    "prob_ft_transformer": "FT-Transformer",
    "prob_tab_transformer": "TabTransformer",
    "prob_saint": "SAINT",
}).round(3))

print("Mean predicted P(churn) by actual label")
display(
    preds.groupby("Churn")[list(prob_map.values())].mean().round(3).rename(columns={v: k for k, v in prob_map.items()})
)


## 8. How to read the comparison

On IBM Telco (7k rows, strong additive signal from contract / tenure / fiber / payment method):

- **Tree ensembles and logistic regression** are the models to beat. They usually win or tie ROC-AUC here.
- **FT-Transformer** is the Transformer to keep: it treats every feature as a token and typically matches the linear/tree band.
- **TabTransformer** is most useful when you care about categorical context; on this table it is a close third among the attention models.
- **SAINT** adds row attention. That helps some datasets; on this one it is in the same band as FT-Transformer. Single-row production scoring should leave inter-sample attention off.

Use **XGBoost or Random Forest** if you need a production classifier tomorrow. Use **FT-Transformer** when you want a tabular deep model that is actually designed for this schema, not a language model.
